# nn-module-subclass composite — cx21: subclass nn.Module and customize __repr__ via extra_repr

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `nn-module-subclass`, `module-extra-repr`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "nn-module-subclass"
DD_ATOM_IDS = ["nn-module-subclass", "module-extra-repr"]
DD_SUBTOPICS = ["PyTorch: nn.Module subclassing", "PyTorch: Module __repr__"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Default `nn.Module.__repr__()` prints `ClassName()` plus the children — which is useless if your class has non-module hyperparameters (kernel size, num features, stride). The **extra_repr** hook is the canonical override:
- **nn-module-subclass** — the usual `class Foo(nn.Module)` skeleton.
- **module-extra-repr** — override `def extra_repr(self) -> str` to return a comma-separated string of the hyperparameters. The base `__repr__` then renders `ClassName(<extra_repr>)`, followed (one indent in) by each registered child's own repr.

Crucially you do NOT override `__repr__` directly — let the base class do the indentation and recursion. You only own the string between the parens.

**Anatomy.**
1. `super().__init__()` + store hyperparams on `self`.
2. `def extra_repr(self) -> str: return f'in_features={self.in_features}, out_features={self.out_features}'`.
3. `repr(m)` produces `'MyLayer(in_features=4, out_features=3)'` — same shape as the built-in `nn.Linear` repr.

### Composite Exercise — subclass nn.Module and customize __repr__ via extra_repr

**Atoms exercised together**: `nn-module-subclass`, `module-extra-repr`

Define a class `MyLayer(nn.Module)` and a builder `cx21_build_my_layer(in_features, out_features)` that returns an instance.

`MyLayer.__init__` must:
1. Call `super().__init__()`.
2. Store `self.in_features = in_features` and `self.out_features = out_features`.

`MyLayer.extra_repr(self)` must return EXACTLY the string `f'in_features={self.in_features}, out_features={self.out_features}'` — no leading/trailing spaces, no parentheses, no class name.

Do NOT define `__repr__` yourself — let the base class wrap your `extra_repr()` into `'MyLayer(<extra_repr>)'`.

`MyLayer.forward(self, x)` returns `x` unchanged (identity — this drill is about the repr, not the math).

In [ ]:
class MyLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        # Atom A (nn-module-subclass): wire up the base bookkeeping FIRST.
        super().__init__()
        # Plain Python attrs — these are the hyperparameters extra_repr will print.
        self.in_features = in_features
        self.out_features = out_features

    def extra_repr(self) -> str:
        # Atom B (module-extra-repr): own ONLY the string between the parens.
        # nn.Module.__repr__ wraps this into f'{ClassName}({extra_repr()})' and recurses
        # into children for us.
        return f'in_features={self.in_features}, out_features={self.out_features}'

    def forward(self, x):
        return x


def cx21_build_my_layer(in_features: int, out_features: int) -> 'MyLayer':
    return MyLayer(in_features, out_features)


<details><summary>Show solution — cx21</summary>

```python
class MyLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        # Atom A (nn-module-subclass): wire up the base bookkeeping FIRST.
        super().__init__()
        # Plain Python attrs — these are the hyperparameters extra_repr will print.
        self.in_features = in_features
        self.out_features = out_features

    def extra_repr(self) -> str:
        # Atom B (module-extra-repr): own ONLY the string between the parens.
        # nn.Module.__repr__ wraps this into f'{ClassName}({extra_repr()})' and recurses
        # into children for us.
        return f'in_features={self.in_features}, out_features={self.out_features}'

    def forward(self, x):
        return x


def cx21_build_my_layer(in_features: int, out_features: int) -> 'MyLayer':
    return MyLayer(in_features, out_features)
```

Overriding `__repr__` directly is the wrong move — you lose the recursion into children and the indentation. `extra_repr` is the surgical override slot: it composes with the base class's tree-printing machinery instead of replacing it.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx21'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx21',
        'subtopics': ["PyTorch: nn.Module subclassing", "PyTorch: Module __repr__"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()